In [8]:
!pip install gliner2

In [6]:
from gliner2 import AutoExtractor

model = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1",)

/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/gliner2/models/boundary/model.py:2101: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)
/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/aau/Desktop/swener_1800/.venv/lib/python3.11/site-packages/gliner2/models/boundary/model.py:1147: RuntimeWarning: Encoder rejected attn_implementation='sdpa'; falling back to 'eager' (DebertaV2Model does not support an attention implementation through torch.nn.functional.scaled_dot_product_attention yet. Please request the support for this architecture: https://github.com/huggingface/transformers/issues/28005. If you believe this error is a bug, please open an issue in Transformers GitHub repos

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("Riksarkivet/swener_1800", split="eval")
ds

Dataset({
    features: ['text', 'entity_types', 'entity_start_chars', 'entity_end_chars', 'id', 'word_start_chars', 'word_end_chars'],
    num_rows: 170
})

In [4]:
from collections import defaultdict
import pandas as pd

def evaluate(ds, preds, labels=None, merge_org=None, name=""):
    """
    ds     : eval split of Riksarkivet/swener_1800
    preds  : iterable of (doc_id, start, end, label); char offsets into the original text
    labels : labels to score, e.g. ["PER", "LOC", "ORG"] for ScandiNER.
             None = every label in gt (after ORG merge). Predictions outside this set are dropped.
    """
    gt = set()
    if merge_org is None:  
        merge_org = labels is not None and "ORG" in labels
    norm = lambda l: "ORG" if merge_org and l.startswith("ORG") else l
    for doc_id, types, starts, ends in zip(
            ds["id"], ds["entity_types"], ds["entity_start_chars"], ds["entity_end_chars"]
        ):
            for t, s, e in zip(types, starts, ends):
                gt.add((doc_id, s, e, norm(t)))

    pred = {(d, s, e, norm(l)) for d, s, e, l in preds}

    keep = set(labels) if labels else {x[3] for x in gt}
    gt = {x for x in gt if x[3] in keep}
    pred = {x for x in pred if x[3] in keep}

    def index(spans):
        d = defaultdict(list)
        for doc, s, e, lab in spans:
            d[doc].append((s, e, lab))
        return d

    gi, pi = index(gt), index(pred)

    def hit(item, idx):  # overlaps at least one span with the same label
        doc, s, e, lab = item
        return any(s < e2 and s2 < e and lab2 == lab for s2, e2, lab2 in idx[doc])

    def prf(tp_p, n_p, tp_r, n_g):
        p = tp_p / n_p if n_p else 0
        r = tp_r / n_g if n_g else 0
        f = 2 * p * r / (p + r) if p + r else 0
        return round(p, 3), round(r, 3), round(f, 3)

    rows = []
    for lab in sorted(keep) + ["micro"]:
        g = {x for x in gt if lab == "micro" or x[3] == lab}
        p = {x for x in pred if lab == "micro" or x[3] == lab}
        strict = prf(len(g & p), len(p), len(g & p), len(g))
        relaxed = prf(sum(hit(x, gi) for x in p), len(p),
                      sum(hit(x, pi) for x in g), len(g))
        rows.append((lab, len(g), len(p), *strict, *relaxed))

    res = pd.DataFrame(rows, columns=["label", "gt", "pred",
                                      "P_strict", "R_strict", "F1_strict",
                                      "P_relax", "R_relax", "F1_relax"])
    per_label = res[res.label != "micro"]
    print(f"== {name} | labels: {len(keep)} | "
          f"macro F1 strict {per_label.F1_strict.mean():.3f} / relaxed {per_label.F1_relax.mean():.3f}")
    display(res)
    return res

## WITH ALL LABELS USING ACRONYMS

In [9]:
from tqdm.auto import tqdm

LABELS = {
    "PER": "person", "LOC": "location", "OCC": "occupation",
    "TME-DATE": "date", "TME-TIME": "time expression", "TME-INTRV": "time interval",
    "MSR-MON": "money amount", "MSR-WEI": "weight", "MSR-VOL": "volume",
    "MSR-LEN": "length", "MSR-DIST": "distance", "MSR-AREA": "area",
    "MSR-OTH": "other measurement",
    "ORG-INST": "institution", "ORG-COMP": "company", "ORG-OTH": "other organization",
    "EVN": "event", "WRK": "work of art or product name", "SYMP": "symptom",
}
NAME_TO_CODE = {v: k for k, v in LABELS.items()}

gl_preds = set()  # (doc_id, start, end, dataset_label)
for doc_id, text in tqdm(zip(ds["id"], ds["text"]), total=len(ds)):
    out = model.extract_entities_long(
        text,
        list(LABELS.values()),
        chunk_size=128,
        chunk_overlap=32,
        include_spans=True,
    )
    for name, ents in out["entities"].items():
        for e in ents:
            gl_preds.add((doc_id, e["start"], e["end"], NAME_TO_CODE[name]))

print(len(gl_preds))
pd.Series([p[3] for p in gl_preds]).value_counts()

100%|██████████| 170/170 [01:33<00:00,  1.82it/s]

7752


PER          3107
LOC          2478
OCC           660
TME-DATE      617
MSR-MON       486
TME-INTRV     177
MSR-VOL        56
SYMP           36
MSR-LEN        23
ORG-COMP       23
TME-TIME       20
EVN            17
ORG-INST       14
MSR-DIST       10
MSR-WEI        10
WRK             8
MSR-AREA        5
MSR-OTH         5
Name: count, dtype: int64

In [13]:
res_gliner = evaluate(ds, gl_preds, name="GLiNER2.5 all labels")

== GLiNER2.5 all labels | labels: 19 | macro F1 strict 0.199 / relaxed 0.316


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,EVN,60,17,0.059,0.017,0.026,0.059,0.017,0.026
1,LOC,1350,2478,0.342,0.627,0.443,0.571,0.895,0.697
2,MSR-AREA,12,5,0.600,0.250,0.353,0.600,0.250,0.353
3,MSR-DIST,8,10,0.200,0.250,0.222,0.500,0.625,0.556
4,MSR-LEN,9,23,0.043,0.111,0.062,0.174,0.556,0.265
5,MSR-MON,471,486,0.521,0.537,0.529,0.681,0.688,0.684
6,MSR-OTH,20,5,0.000,0.000,0.000,0.000,0.000,0.000
7,MSR-VOL,47,56,0.214,0.255,0.233,0.286,0.340,0.311
8,MSR-WEI,15,10,0.400,0.267,0.320,0.500,0.333,0.400
9,OCC,1065,660,0.641,0.397,0.490,0.691,0.427,0.528


## ONLY WITH BIOS LABELS AS WITH SCANDINER

In [11]:
from tqdm.auto import tqdm

LABELS = {
    "PER": "person", "LOC": "location", "ORG": "organisation",
}
NAME_TO_CODE = {v: k for k, v in LABELS.items()}

gl_preds_bios = set()  # (doc_id, start, end, dataset_label)
for doc_id, text in tqdm(zip(ds["id"], ds["text"]), total=len(ds)):
    out = model.extract_entities_long(
        text,
        list(LABELS.values()),
        chunk_size=128,
        chunk_overlap=32,
        include_spans=True,
    )
    for name, ents in out["entities"].items():
        for e in ents:
            gl_preds_bios.add((doc_id, e["start"], e["end"], NAME_TO_CODE[name]))

print(len(gl_preds_bios))
pd.Series([p[3] for p in gl_preds_bios]).value_counts()

100%|██████████| 170/170 [01:22<00:00,  2.07it/s]

5041


PER    2797
LOC    1758
ORG     486
Name: count, dtype: int64

In [12]:
res_glinerBIOs = evaluate(ds, gl_preds_bios, labels=["PER", "LOC", "ORG"], name="GLiNER2.5 PER/LOC/ORG")

== GLiNER2.5 PER/LOC/ORG | labels: 3 | macro F1 strict 0.466 / relaxed 0.666


,label,gt,pred,P_strict,R_strict,F1_strict,P_relax,R_relax,F1_relax
0,LOC,1350,1758,0.458,0.597,0.519,0.700,0.832,0.760
1,ORG,455,486,0.379,0.404,0.391,0.447,0.477,0.461
2,PER,2488,2797,0.460,0.517,0.487,0.730,0.828,0.776
3,micro,4293,5041,0.451,0.530,0.488,0.692,0.792,0.739
